In [1]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np

In [3]:
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/25 14:09:38 WARN Utils: Your hostname, kloc, resolves to a loopback address: 127.0.1.1; using 10.64.133.26 instead (on interface wlan0)
25/10/25 14:09:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/25 14:09:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [16]:
countries = spark.read.csv("countries of the world.csv",inferSchema=True,header=True)

In [18]:
countries.toPandas()

,Country,Region,Population,Area (sq. mi.),Pop. Density (per sq. mi.),Coastline (coast/area ratio),Net migration,Infant mortality (per 1000 births),GDP ($ per capita),Literacy (%),Phones (per 1000),Arable (%),Crops (%),Other (%),Climate,Birthrate,Deathrate,Agriculture,Industry,Service
0,Afghanistan,ASIA (EX. NEAR EAST),31056997,647500,"48,0","0,00","23,06","163,07",700.0,"36,0","3,2","12,13","0,22","87,65",1,"46,6","20,34","0,38","0,24","0,38"
1,Albania,EASTERN EUROPE,3581655,28748,"124,6","1,26","-4,93","21,52",4500.0,"86,5","71,2","21,09","4,42","74,49",3,"15,11","5,22","0,232","0,188","0,579"
2,Algeria,NORTHERN AFRICA,32930091,2381740,"13,8","0,04","-0,39",31,6000.0,"70,0","78,1","3,22","0,25","96,53",1,"17,14","4,61","0,101","0,6","0,298"
3,American Samoa,OCEANIA,57794,199,"290,4","58,29","-20,71","9,27",8000.0,"97,0","259,5",10,15,75,2,"22,46","3,27",None,None,None
4,Andorra,WESTERN EUROPE,71201,468,"152,1","0,00","6,6","4,05",19000.0,"100,0","497,2","2,22",0,"97,78",3,"8,71","6,25",None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,West Bank,NEAR EAST,2460492,5860,"419,9","0,00","2,98","19,62",800.0,None,"145,2","16,9","18,97","64,13",3,"31,67","3,92","0,09","0,28","0,63"
223,Western Sahara,NORTHERN AFRICA,273008,266000,"1,0","0,42",None,None,NaN,None,None,"0,02",0,"99,98",1,None,None,None,None,"0,4"
224,Yemen,NEAR EAST,21456188,527970,"40,6","0,36",0,"61,5",800.0,"50,2","37,2","2,78","0,24","96,98",1,"42,89","8,3","0,135","0,472","0,393"
225,Zambia,SUB-SAHARAN AFRICA,11502010,752614,"15,3","0,00",0,"88,29",800.0,"80,6","8,2","7,08","0,03","92,9",2,41,"19,93","0,22","0,29","0,489"


In [9]:
from pyspark.sql.types import FloatType
from pyspark.sql.functions import udf
float_udf = udf(lambda s: float(s.replace(',','.')), FloatType())

In [27]:
# countries.withColumn("Area (sq. mi.)", float_udf("Area (sq. mi.)"))
countries.withColumn("Industry", float_udf("Industry"))

DataFrame[Country: float, Region: float, Population: float, Area (sq. mi.): int, Pop. Density (per sq. mi.): string, Coastline (coast/area ratio): string, Net migration: string, Infant mortality (per 1000 births): string, GDP ($ per capita): int, Literacy (%): string, Phones (per 1000): string, Arable (%): string, Crops (%): string, Other (%): string, Climate: string, Birthrate: string, Deathrate: string, Agriculture: string, Industry: float, Service: string]

In [21]:
countries.columns

['Country',
 'Region',
 'Population',
 'Area (sq. mi.)',
 'Pop. Density (per sq. mi.)',
 'Coastline (coast/area ratio)',
 'Net migration',
 'Infant mortality (per 1000 births)',
 'GDP ($ per capita)',
 'Literacy (%)',
 'Phones (per 1000)',
 'Arable (%)',
 'Crops (%)',
 'Other (%)',
 'Climate',
 'Birthrate',
 'Deathrate',
 'Agriculture',
 'Industry',
 'Service']

In [23]:
from pyspark.sql.functions import col, regexp_replace

cols_to_convert = ['Agriculture', 'Industry', 'Service']

for col_name in cols_to_convert:
    print(col_name)
    countries = countries.withColumn(
        col_name,
        float_udf(col_name)
    )
countries.printSchema()

{"ts": "2025-10-25 13:06:03.473", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Area (sq`.` mi`.`)` cannot be resolved. Did you mean one of the following? [`Area (sq. mi.)`, `Arable (%)`, `Crops (%)`, `GDP ($ per capita)`, `Net migration`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o148.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Area (sq`.` mi`.`)` cannot be resolved. Did you mean one of the following? [`Area (sq. mi.)`, `Arable (%)`, `Crops (%)`, `GDP ($ per capita)`, `Net migration`]. SQLSTATE: 42703;\n'Project [Country#183, Region#185, Pop

Country
Region
Population
Area (sq. mi.)


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Area (sq`.` mi`.`)` cannot be resolved. Did you mean one of the following? [`Area (sq. mi.)`, `Arable (%)`, `Crops (%)`, `GDP ($ per capita)`, `Net migration`]. SQLSTATE: 42703;
'Project [Country#183, Region#185, Population#187, <lambda>('Area (sq. mi.))#188 AS Area (sq. mi.)#189, Pop. Density (per sq. mi.)#135, Coastline (coast/area ratio)#136, Net migration#137, Infant mortality (per 1000 births)#138, GDP ($ per capita)#139, Literacy (%)#140, Phones (per 1000)#141, Arable (%)#142, Crops (%)#143, Other (%)#144, Climate#145, Birthrate#146, Deathrate#147, Agriculture#148, Industry#149, Service#150]
+- Project [Country#183, Region#185, <lambda>(Population#179)#186 AS Population#187, Area (sq. mi.)#134, Pop. Density (per sq. mi.)#135, Coastline (coast/area ratio)#136, Net migration#137, Infant mortality (per 1000 births)#138, GDP ($ per capita)#139, Literacy (%)#140, Phones (per 100